# bio-rag-eval — interactive demo

This notebook walks through the bio-rag-eval pipeline using the
`MockJudge` so it runs offline with no API key. Replace the mock
with `AnthropicJudge(...)` or `OpenAIJudge(...)` for a real eval.

Steps:
1. Load one curated case from `examples/sample_gold_standards/`
2. Load the matching agent response from `examples/sample_responses.jsonl`
3. Run a single-case eval and inspect the per-claim grounding judgments
4. Inspect the bootstrap CIs on the aggregate metrics
5. Show the markdown report

In [ ]:
import sys
from pathlib import Path

# Make `src/` importable when running from a checkout.
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

from bio_rag_eval.judges import MockJudge
from bio_rag_eval.report import render_markdown
from bio_rag_eval.runner import EvalRunner, RunConfig, load_cases_from_yaml
from bio_rag_eval.schemas.case import AgentResponse
from bio_rag_eval.schemas.claims import Claim, ClaimList, ClaimType
from bio_rag_eval.schemas.judgments import (
    CompletenessJudgment, EvidenceSufficiency, GroundingJudgment,
    GroundingLabel, MechanismJudgment,
)

## 1. Load the SOD1-ALS case + agent response

In [ ]:
case = load_cases_from_yaml([str(ROOT / 'examples' / 'sample_gold_standards' / 'sod1_als.yaml')])[0]
print(case.case_id, '-', case.name)
print('expected target:', case.expected_target)
print('expected modulation:', case.expected_modulation)
print('facets:', len(case.expected_facets))

In [ ]:
responses_path = ROOT / 'examples' / 'sample_responses.jsonl'
response = None
for line in responses_path.read_text(encoding='utf-8').splitlines():
    line = line.strip()
    if not line:
        continue
    r = AgentResponse.model_validate_json(line)
    if r.case_id == case.case_id:
        response = r
        break
print('answer (first 240 chars):')
print(response.answer[:240])
print(f'\ncitations: {len(response.citations)}')

## 2. Build a MockJudge

The mock just returns canned pydantic objects per schema — no LLM,
no network. Swap with `AnthropicJudge(model='claude-opus-4-7')` for a real run.

In [ ]:
judge = MockJudge({
    ClaimList: ClaimList(claims=[
        Claim(claim_id='c1', text='SOD1 mutations cause toxic gain-of-function.', claim_type=ClaimType.FACTUAL, cited_ids=['s1']),
        Claim(claim_id='c2', text='Tofersen reduced CSF NfL by ~60% at week 28.', claim_type=ClaimType.QUANTITATIVE, cited_ids=['s2']),
        Claim(claim_id='c3', text='Tofersen received FDA accelerated approval in April 2023.', claim_type=ClaimType.FACTUAL, cited_ids=['s3']),
    ]),
    GroundingJudgment: [
        GroundingJudgment(claim_id='c1', label=GroundingLabel.SUPPORTED, rationale='snippet explicitly states this.', confidence=0.95),
        GroundingJudgment(claim_id='c2', label=GroundingLabel.SUPPORTED, rationale='snippet states ~60%.', confidence=0.95),
        GroundingJudgment(claim_id='c3', label=GroundingLabel.SUPPORTED, rationale='snippet confirms April 2023.', confidence=0.95),
    ],
    MechanismJudgment: MechanismJudgment(score=5, rationale='Full causal chain from variant to ASO.'),
    EvidenceSufficiency: EvidenceSufficiency(score=4, n_independent_sources=2, has_primary_literature=True, has_human_data=True, rationale='Two NEJM trials plus the FDA letter.'),
    CompletenessJudgment: [CompletenessJudgment(facet=f, covered=True, rationale='covered') for f in case.expected_facets],
})

## 3. Run the eval

In [ ]:
runner = EvalRunner(judge=judge, config=RunConfig(seed=7, run_bias_check=False, citation_offline=True))
report = runner.run([case], [response])
cr = report.case_results[0]
print('flags:', cr.flags or '(none)')
for k, v in sorted(cr.metrics.items()):
    print(f'  {k:<35s} {v}')

## 4. Inspect per-claim judgments

In [ ]:
for c in cr.extracted_claims:
    j = next((j for j in cr.grounding_judgments if j.claim_id == c.claim_id), None)
    label = j.label.value if j else 'no-judgment'
    print(f'[{c.claim_id}] {c.claim_type.value:<14s} {label:<22s} {c.text[:80]}')

## 5. Show the markdown report

In [ ]:
from IPython.display import Markdown
Markdown(render_markdown(report))

## Next steps

- Swap `MockJudge` for `AnthropicJudge(model='claude-opus-4-7')` to run
  against a real LLM judge.
- Add `RunConfig(run_bias_check=True)` to also run the swapped-rubric
  re-grade and surface position-bias metrics.
- For larger studies, run the whole 10-case set via
  `examples/eval_therapy_agent.py`.